## Python Libraries Imports


In [ ]:
#Imports all the needed python libraries
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# HF = human-vs-AI-generated dataset from Hugging Face (ahmadreza13/human-vs-Ai-generated-dataset)
# CB = dataset from the M-DAIGT (Multi-Domain Detection of AI-Generated Text) shared task,
# hosted on CodaBench. Not redistributed here — see README for source and citation.


## LOADING ALL THE DATASETS

In [ ]:
#Loading the HF dataset
hf_ds = load_dataset("ahmadreza13/human-vs-Ai-generated-dataset")

In [ ]:
#Preprocessing HF dataset
hf_df = hf_ds["train"].to_pandas()

#Splits the table into human/AI groups according to the Label
human_df = hf_df[hf_df["generated"] == 0]
ai_df    = hf_df[hf_df["generated"] == 1]

#Takes 10k sample from each group, ensuring near-equal group size 
human_sample = human_df.sample(n=10000, random_state=1)
ai_sample    = ai_df.sample(n=10000, random_state=1)

#Cleans the dataset by removing duplicates in each group, ensuring reduction in data leakage
human_clean = human_sample.drop_duplicates(subset=["data"]).reset_index(drop=True)
ai_clean    = ai_sample.drop_duplicates(subset=["data"]).reset_index(drop=True)

#Joins the group of datas together, and de-duplicates it once more, incase there are any duplicates shared within the groups.
hf_balanced_df = pd.concat([human_clean, ai_clean], ignore_index=True)
hf_balanced_df = hf_balanced_df.drop_duplicates(subset=["data"]).reset_index(drop=True)

#Reshuffles the dataset, to ensure AI/Human text are in random order. To reduce possibility of Model learning shortcut
hf_balanced_df = hf_balanced_df.sample(frac=1, random_state=1).reset_index(drop=True)



In [ ]:
#Loading CB dataset
cb_ds1 = pd.read_csv("task1-train.csv")
cb_ds2 = pd.read_csv("task2-train.csv")

#Joining both parts of CB dataset together and dropping the duplicates  
cb_all_ds = pd.concat([cb_ds1, cb_ds2], ignore_index=True)
cb_all_ds = cb_all_ds.drop_duplicates(subset="text").reset_index(drop=True)
#Mapping CB dataset label value to binary 
cb_all_ds["label"] = cb_all_ds["label"].map({
    "human": 0,
    "machine": 1
})

In [ ]:
#Creates a copy object of HF and CB columns
hf_copy = hf_balanced_df[["data", "generated"]].copy()
cb_copy = cb_all_ds[["text", "label"]].copy()

#HF copy columns are renamed  to match CB columns
hf_copy.columns = ["text", "label"]

#HF and CB copies are stacked together to create a big combined dataset, following another duplicate drop check
combined_ds = pd.concat([hf_copy, cb_copy], ignore_index=True)
combined_ds = combined_ds.drop_duplicates(subset="text").reset_index(drop=True)

## HF->HF MODEL TRAINING  AND TESTING

In [ ]:
#Stores HF text sample
hf_x = hf_balanced_df["data"]
#Stores HF sample label
hf_y = hf_balanced_df["generated"]

In [ ]:
#Splits the HF training and test data, with a  80/20 split
hf_train_x, hf_test_x, hf_train_y, hf_test_y = train_test_split (
    hf_x,hf_y,
    test_size = 0.2,
    random_state=1
)

In [ ]:
#TF-IDF vectoriser is called to learn and convert the training text data, and then also the test data
indomain_vector_function = TfidfVectorizer()
hf_tfidf_train_x = indomain_vector_function.fit_transform(hf_train_x)
hf_tfidf_test_x = indomain_vector_function.transform(hf_test_x)

In [ ]:
#creates the log regression model for this test 
log_model = LogisticRegression(max_iter=1000)
#The log regression models is training on the vectorised training data
log_model.fit(hf_tfidf_train_x, hf_train_y)
#The log model is now tested on the vectorised test data
hf_log_data_prediction = log_model.predict(hf_tfidf_test_x)

#print results
print("Accuracy Score", accuracy_score(hf_test_y,hf_log_data_prediction))
print(classification_report(hf_test_y, hf_log_data_prediction, digits = 4))
print(confusion_matrix(hf_test_y, hf_log_data_prediction))

In [ ]:
#creates the LinearSVC model for this test 
SVC_model= LinearSVC(max_iter = 1000, verbose = 1)
#The LinearSVC model is trained on the vectorised training data
SVC_model.fit(hf_tfidf_train_x, hf_train_y)
#The LinearSVC is now tested on the vectorised test data
hf_SVC_data_prediction = SVC_model.predict(hf_tfidf_test_x)

#print results
print("Accuracy Score", accuracy_score(hf_test_y, hf_SVC_data_prediction))
print(classification_report(hf_test_y, hf_SVC_data_prediction, digits = 4))
print(confusion_matrix(hf_test_y, hf_SVC_data_prediction))

## CB -> CB TRAINING AND TESTING

In [ ]:
#Store CB text sample 
cb_x = cb_all_ds["text"]
#Store CB sample labels
cb_y = cb_all_ds["label"]

In [ ]:
#Splits the CB training and test data, with a  80/20 split
cb_train_x, cb_test_x, cb_train_y, cb_test_y = train_test_split(
    cb_x, cb_y,
    test_size=0.2,
    random_state=1,
    stratify=cb_y
)

In [ ]:
##TF-IDF vectoriser is called to learn and convert the training text data, and then also the test data
cb_tfidf_train_x = indomain_vector_function.fit_transform(cb_train_x)
cb_tfidf_test_x = indomain_vector_function.transform(cb_test_x)

In [ ]:
#Log regression model is trained on vectorised CB data 
log_model.fit(cb_tfidf_train_x, cb_train_y)
#log regression is now tested on vectorised CB data =
cb_data_prediction = log_model.predict(cb_tfidf_test_x)

#Prints results
print("Accuracy Score", accuracy_score(cb_test_y, cb_data_prediction))
print(classification_report(cb_test_y, cb_data_prediction, digits = 4))
print(confusion_matrix(cb_test_y, cb_data_prediction))

In [ ]:
#LinearSVC is trained on vectorised CB train data
SVC_model.fit(cb_tfidf_train_x, cb_train_y)
#LinearSVC is tested on vectorised CB test data
SVC_cb_data_prediction = SVC_model.predict(cb_tfidf_test_x)

#Prints results
print("Accuracy Score", accuracy_score(cb_test_y, SVC_cb_data_prediction))
print(classification_report(cb_test_y, SVC_cb_data_prediction, digits = 4))
print(confusion_matrix(cb_test_y, SVC_cb_data_prediction))

## COMBO -> COMBO TRAINING AND TESTING

In [ ]:
#store combined sample data
combined_x = combined_ds["text"]
#store combined sample label
combined_y = combined_ds["label"]

In [ ]:
#Splits the combined training and test data, with a  80/20 split
combined_train_x, combined_test_x, combined_train_y, combined_test_y = train_test_split (
    combined_x,combined_y,
    test_size = 0.2,
    random_state=1,
    stratify=combined_y
)

In [ ]:
combined_tfidf_train_x = indomain_vector_function.fit_transform(combined_train_x)
combined_tfidf_test_x = indomain_vector_function.transform(combined_test_x)

In [ ]:
#log regression model is trained and tested on combined dataset
log_model.fit(combined_tfidf_train_x, combined_train_y)
combined_data_prediction = log_model.predict(combined_tfidf_test_x)

print("Accuracy Score", accuracy_score(combined_test_y, combined_data_prediction))
print(classification_report(combined_test_y, combined_data_prediction, digits = 4))
print(confusion_matrix(combined_test_y, combined_data_prediction))

In [ ]:
#linearSVC model is trained and tested on combined dataset
SVC_model.fit(combined_tfidf_train_x, combined_train_y)
SVC_combined_data_prediction = SVC_model.predict(combined_tfidf_test_x)

print("Accuracy Score", accuracy_score(combined_test_y, SVC_combined_data_prediction))
print(classification_report(combined_test_y, SVC_combined_data_prediction, digits = 4))
print(confusion_matrix(combined_test_y, SVC_combined_data_prediction))

## HF->CB TRAINING AND TESTING

In [ ]:
#Creates new x and y hf values for cross-domain settings 
cross_hf_x = hf_copy["text"]
cross_hf_y = hf_copy["label"]

In [ ]:
#Creates new x and y cb values for cross-domain settings
cross_cb_x = cb_copy["text"]
cross_cb_y = cb_copy["label"]

In [ ]:
#Creates new cross-domain hf train and test splits 
cross_train_x_hf, cross_test_x_hf, cross_train_y_hf, cross_test_y_hf = train_test_split(
    cross_hf_x, cross_hf_y,
    test_size=0.2, 
    random_state=1, 
    stratify=cross_hf_y
)

In [ ]:
#Creates new cross-domain cb train and test splits 
cross_train_x_cb, cross_test_x_cb, cross_train_y_cb, cross_test_y_cb = train_test_split(
    cross_cb_x, cross_cb_y, 
    test_size=0.2, 
    random_state=1, 
    stratify=cross_cb_y
)

In [ ]:
#Creates a new cross-domain vectoriser to learn to vectorise on hf train set, and vectorise on cb test set
cross_vector_function_hf = TfidfVectorizer()
cross_tfidf_train_hf = cross_vector_function_hf.fit_transform(cross_train_x_hf)
cross_tfidf_test_cb = cross_vector_function_hf.transform(cross_test_x_cb)

In [ ]:
#Log regression model is trained on hf dataset
log_model.fit(cross_tfidf_train_hf,cross_train_y_hf)
#Log regression performs test on cb dataset
hftocb_log_predictor = log_model.predict(cross_tfidf_test_cb)

print("Accuracy Score", accuracy_score(cross_test_y_cb, hftocb_log_predictor))
print(classification_report(cross_test_y_cb, hftocb_log_predictor, digits = 4))
print(confusion_matrix(cross_test_y_cb, hftocb_log_predictor))

In [ ]:
#LinearSVC model is trained on hf dataset
SVC_model.fit(cross_tfidf_train_hf,cross_train_y_hf)
#LinearSVC model performs test on cb dataset
hftocb_SVC_predictor = SVC_model.predict(cross_tfidf_test_cb)

print("Accuracy Score", accuracy_score(cross_test_y_cb, hftocb_SVC_predictor))
print(classification_report(cross_test_y_cb, hftocb_SVC_predictor, digits = 4))
print(confusion_matrix(cross_test_y_cb, hftocb_SVC_predictor))

## CB->HF TRAINING AND TESTING 

In [ ]:
#TF-IDF vectoriser learning to vectorise with cb train data, which is then used to vectorise hf test data
cross_vector_function_cb = TfidfVectorizer()
cross_tfidf_train_cb = cross_vector_function_cb.fit_transform(cross_train_x_cb)
cross_tfidf_test_hf = cross_vector_function_cb.transform(cross_test_x_hf)

In [ ]:
#log regression model learning from dataset cb
log_model.fit(cross_tfidf_train_cb,cross_train_y_cb)
#log regression model performs test on dataset hf
cbtohf_log_predictor = log_model.predict(cross_tfidf_test_hf)

print("Accuracy Score", accuracy_score(cross_test_y_hf, cbtohf_log_predictor))
print(classification_report(cross_test_y_hf, cbtohf_log_predictor, digits = 4))
print(confusion_matrix(cross_test_y_hf, cbtohf_log_predictor))

In [ ]:
#LinearSVC model learning from dataset cb
SVC_model.fit(cross_tfidf_train_cb,cross_train_y_cb)
#LinearSVC model performs test on dataset hf
cbtohf_SVC_predictor = SVC_model.predict(cross_tfidf_test_hf)

print("Accuracy Score", accuracy_score(cross_test_y_hf, cbtohf_SVC_predictor))
print(classification_report(cross_test_y_hf, cbtohf_SVC_predictor, digits = 4))
print(confusion_matrix(cross_test_y_hf, cbtohf_SVC_predictor))

## COMBO->HF TRAINING AND TESTING (DOMAIN ADAPTATION)

In [ ]:
#tf-idf vectoriser for combined cross-domain testing
cross_vector_function_combo = TfidfVectorizer()
tfidf_train_combo = cross_vector_function_combo.fit_transform(combined_train_x)

#vectoriser converts hf and cb test data into numbers
tfidf_test_hf_from_combo = cross_vector_function_combo.transform(cross_test_x_hf)
tfidf_test_cb_from_combo = cross_vector_function_combo.transform(cross_test_x_cb)

In [ ]:
#Log regression model trained on combined dataset, and tested on HF dataset
log_model.fit(tfidf_train_combo, combined_train_y)
combo_to_hf_log_preds = log_model.predict(tfidf_test_hf_from_combo)

print("Accuracy Score:", accuracy_score(cross_test_y_hf, combo_to_hf_log_preds))
print(classification_report(cross_test_y_hf, combo_to_hf_log_preds, digits=4))
print(confusion_matrix(cross_test_y_hf, combo_to_hf_log_preds))

In [ ]:
#LinearSVC model trained on combined dataset, and tested on HF dataset
SVC_model.fit(tfidf_train_combo, combined_train_y)
combo_to_hf_svc_preds = SVC_model.predict(tfidf_test_hf_from_combo)

print("Accuracy Score:", accuracy_score(cross_test_y_hf, combo_to_hf_svc_preds))
print(classification_report(cross_test_y_hf, combo_to_hf_svc_preds, digits=4))
print(confusion_matrix(cross_test_y_hf, combo_to_hf_svc_preds))

## COMBO->CB TRAINING AND TESTING (DOMAIN ADAPTATION)

In [ ]:
#Log regression model trained on combined dataset, and tested on CB dataset
log_model.fit(tfidf_train_combo, combined_train_y)
combo_to_cb_log_preds = log_model.predict(tfidf_test_cb_from_combo)

print("Accuracy Score:", accuracy_score(cross_test_y_cb, combo_to_cb_log_preds))
print(classification_report(cross_test_y_cb, combo_to_cb_log_preds, digits=4))
print(confusion_matrix(cross_test_y_cb, combo_to_cb_log_preds))

In [ ]:
#LinearSVC model trained on combined dataset, and tested on CB dataset
SVC_model.fit(tfidf_train_combo, combined_train_y)
combo_to_cb_svc_preds = SVC_model.predict(tfidf_test_cb_from_combo)

print("Accuracy Score:", accuracy_score(cross_test_y_cb, combo_to_cb_svc_preds))
print(classification_report(cross_test_y_cb, combo_to_cb_svc_preds, digits=4))
print(confusion_matrix(cross_test_y_cb, combo_to_cb_svc_preds))